# Ekstraksi Data Pengguna

## data and init

In [1]:
import pandas as pd
import numpy as np
from itables import show

In [2]:
trans = pd.read_csv('data/processed/transactions_enriched.csv', delimiter=';', low_memory=False)

In [3]:
trans.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 5372 entries, 0 to 5371
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   ID Anggota          5372 non-null   int64
 1   Nama Anggota        5372 non-null   str  
 2   Kode Eksemplar      5372 non-null   int64
 3   Judul               5372 non-null   str  
 4   Tanggal Pinjam      5372 non-null   str  
 5   tahun_pinjam        5372 non-null   int64
 6   bulan_pinjam        5372 non-null   int64
 7   bulan_tahun_pinjam  5372 non-null   str  
 8   book_id             5372 non-null   str  
dtypes: int64(4), str(5)
memory usage: 830.7 KB


In [4]:
show(trans)

Loading ITables v2.7.0 from the internet... (need help?)


## clean id anggota

In [5]:
#convert id to integer
trans['ID Anggota'] = (
    trans['ID Anggota']
    .astype(str)
    .str.replace(',', '.', regex=False)
)

trans['ID Anggota'] = pd.to_numeric(trans['ID Anggota'], errors='coerce')

In [6]:
trans_cleaned = trans.dropna(subset=['ID Anggota'])
trans_cleaned['ID Anggota'] = trans_cleaned['ID Anggota'].astype('Int64')
trans_cleaned

,ID Anggota,Nama Anggota,Kode Eksemplar,Judul,Tanggal Pinjam,tahun_pinjam,bulan_pinjam,bulan_tahun_pinjam,book_id
0,196903132021211000,"Fatchullah, S.Sos, M.A",91185703,Kalkulus buku 1,2023-09-08,2023,9,2023-09,B020594
1,20081010235,20081010235,91185703,Kalkulus buku 1,2023-09-08,2023,9,2023-09,B020594
2,196903132021211000,"Fatchullah, S.Sos, M.A",91185703,Kalkulus buku 1,2023-09-10,2023,9,2023-09,B020594
3,196903132021211000,"Fatchullah, S.Sos, M.A",90916701,Riset Operasi dan Ekonofisika (operation rese...,2023-09-11,2023,9,2023-09,B023857
4,22031010134,DAMARA RAMADHANI MARITZA,90271402,Elements of organic chemistry,2023-09-11,2023,9,2023-09,B028845
...,...,...,...,...,...,...,...,...,...
5367,22013010294,WIDYA FRAHESTIKA,91554701,Akuntansi Keuangan Daerah Berbasis Akrual,2025-11-17,2025,11,2025-11,B018193
5368,23013010026,Zidan Ardiansyah,90366601,"Pengantar ekonomi, jilid 2",2025-11-17,2025,11,2025-11,B024457
5369,22012010010,KANIA LUTHFIYYAH,91028503,Bank and financial institution management conv...,2025-11-18,2025,11,2025-11,B025091
5370,25013010324,ANGGUN ARTIKA DEWI,90884701,Mengenal bebrapa uji statistik dalam penelitian,2025-11-18,2025,11,2025-11,B023280


In [7]:
user_ids = trans_cleaned['ID Anggota'].unique()
user_ids

<IntegerArray>
[196903132021211000,        20081010235,        22031010134,
        22042010081,        22043010108,        23031010070,
        23031010115,        23071010329,        20024010085,
        23031010156,
 ...
        23032010194,        22032010023,        22042010123,
        25041010001,        23032010204,        25084010013,
        22071310360,        22042010044,        25012010392,
        25012010116]
Length: 1798, dtype: Int64

In [8]:
users = trans_cleaned.groupby('ID Anggota')['Nama Anggota'].first().reset_index()
users

,ID Anggota,Nama Anggota
0,2302111080,Kharmelia Putri
1,2399100145,debora padang
2,2399200083,Nabila Nurcahyani
3,17041010008,Leticia Nuzululita Agustine
4,18011010001,Winarti
...,...,...
1793,198706172025062000,"Dr. Eunike Rose Mita Lukiani, M.Pd"
1794,199010282024062000,Nenni Mona Aruan
1795,199312042024061000,Fajar Timur
1796,199509302024062000,Yesi Mustika Ningsih


In [9]:
users_df = pd.DataFrame(users)
users_df

,ID Anggota,Nama Anggota
0,2302111080,Kharmelia Putri
1,2399100145,debora padang
2,2399200083,Nabila Nurcahyani
3,17041010008,Leticia Nuzululita Agustine
4,18011010001,Winarti
...,...,...
1793,198706172025062000,"Dr. Eunike Rose Mita Lukiani, M.Pd"
1794,199010282024062000,Nenni Mona Aruan
1795,199312042024061000,Fajar Timur
1796,199509302024062000,Yesi Mustika Ningsih


In [10]:
users_df.tail(10)

,ID Anggota,Nama Anggota
1788,196701231993032000,Dra Ec TITUK DIAH WIDAYANTIE M AKS
1789,196903132021211000,"Fatchullah, S.Sos, M.A"
1790,197606272021211000,Mochamad Budiyono
1791,197708172021212000,Dyan Agustin
1792,198511062019031000,Aris Sutejo
1793,198706172025062000,"Dr. Eunike Rose Mita Lukiani, M.Pd"
1794,199010282024062000,Nenni Mona Aruan
1795,199312042024061000,Fajar Timur
1796,199509302024062000,Yesi Mustika Ningsih
1797,199901062024062000,Mirza Ramadhani


In [11]:
users_df['ID Anggota'] = users_df['ID Anggota'].astype(str)

users_df['role'] = np.select(
    [
        (users_df['ID Anggota'].str.len() == 18) & (users_df['ID Anggota'].str.isnumeric()),
        (users_df['ID Anggota'].str.len() == 11) & (users_df['ID Anggota'].str.isnumeric())
    ],
    ['lecturer', 'student'],
    default='unknown'
)

show(users_df)

Loading ITables v2.7.0 from the internet... (need help?)


In [12]:
# Format name
users_df['Nama Anggota'] = (users_df['Nama Anggota'].str.title())
show(users_df)

Loading ITables v2.7.0 from the internet... (need help?)


In [13]:
unknown_df = users_df[users_df['role'] == 'unknown']
unknown_df.info()

<class 'pandas.DataFrame'>
Index: 10 entries, 0 to 1784
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ID Anggota    10 non-null     str  
 1   Nama Anggota  10 non-null     str  
 2   role          10 non-null     str  
dtypes: str(3)
memory usage: 740.0 bytes


## Student

In [14]:
students_df = users_df[users_df['role'] == 'student']
students_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1775 entries, 3 to 1777
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ID Anggota    1775 non-null   str  
 1   Nama Anggota  1775 non-null   str  
 2   role          1775 non-null   str  
dtypes: str(3)
memory usage: 108.9 KB


61 => Magister Manajemen,
62 => Magister Akuntansi,
63 => Magister Agroteknologi
64 => Magister Agribisnis
65 => Magister Ilmu Lingkungan
66 => Magister Teknologi Informasi
67 => Magister Ilmu Komunikasi
68 => Magister Hukum

In [15]:
jurusan_map = {
    47: 'Bisnis Proses',
    41: 'Ilmu Administrasi Negara / Publik',
    44: 'Hubungan Internasional',
    42: 'Ilmu Administrasi Niaga / Bisnis',
    43: 'Ilmu Komunikasi',
    46: 'Linguistik Indonesia',
    45: 'Pariwisata',
    13: 'Akuntansi',
    14: 'Kewirausahaan',
    12: 'Manajemen',
    11: 'Ekonomi Pembangunan',
    91: 'Kedokteran',
    71: 'Ilmu Hukum',
    25: 'Agroteknologi',
    24: 'Agribisnis',
    84: 'Bisnis Digital',
    83: 'Sains Data',
    82: 'Sistem Informasi',
    51: 'Arsitektur',
    53: 'Desain Interior',
    81: 'Teknik Informatika',
    32: 'Teknik Industri',
    31: 'Teknik Kimia',
    34: 'Teknik Lingkungan',
    36: 'Teknik Mesin',
    33: 'Teknologi Pangan',
    35: 'Teknik Sipil',
    52: 'Desain Komunikasi Visual',
    37: 'Fisika',
    61: 'Magister Manajemen',
    62: 'Magister Akuntansi',
    63: 'Magister Agroteknologi',
    64: 'Magister Agribisnis',
    65: 'Magister Ilmu Lingkungan',
    66: 'Magister Teknologi Informasi',
    67: 'Magister Ilmu Komunikasi',
    68: 'Magister Hukum',
}

fakultas_map = {
    1: 'Fakultas Ekonomi dan Bisnis',
    2: 'Fakultas Pertanian',
    3: 'Fakultas Teknik dan Sains',
    4: 'Fakultas Ilmu Sosial, Politik, dan Bahasa',
    5: 'Fakultas Arsitektur dan Desain',
    7: 'Fakultas Hukum',
    8: 'Fakultas Ilmu Komputer',
    9: 'Fakultas Kedokteran'
}

prodi_to_fakultas = {
    # FEB
    11:1, 12:1, 13:1, 14:1,
    61:1, 62:1,

    # Pertanian
    24:2, 25:2,
    63:2, 64:2,

    # Teknik & Sains
    31:3, 32:3, 33:3, 34:3, 35:3, 36:3, 37:3,
    65:3,

    # FISIP & Bahasa
    41:4, 42:4, 43:4, 44:4, 45:4, 46:4, 47:4,
    67:4,

    # Arsitektur & Desain
    51:5, 52:5, 53:5,

    # Hukum
    71:7,
    68:7,

    # Ilmu Komputer
    81:8, 82:8, 83:8, 84:8, 66:3,

    # Kedokteran
    91:9,
}

In [16]:
def extract_student_data(id):
    id = id.astype(str)

    angkatan = id.str[:2].astype(int) + 2000
    prodi_id = id.str[3:5].astype(int)
    jenjang_id = id.str[6].astype(int)
    jenjang = ""

    jenjang = jenjang_id.map({
        1: 'S1',
        2: 'S2'
    })

    jurusan = prodi_id.map(jurusan_map)
    fakultas_id = prodi_id.map(prodi_to_fakultas)
    fakultas = fakultas_id.map(fakultas_map)

    return pd.DataFrame({
        'angkatan': angkatan,
        'prodi_id': prodi_id,
        'jurusan': jurusan,
        'fakultas_id': fakultas_id,
        'fakultas': fakultas,
        'jenjang': jenjang
    })


In [17]:
students_df[['angkatan', 'fakultas_id', 'prodi_id', 'jurusan', 'fakultas', 'jenjang']] = extract_student_data(students_df['ID Anggota'])
show(students_df)

Loading ITables v2.7.0 from the internet... (need help?)


In [18]:
students_df.isna().sum()

ID Anggota      0
Nama Anggota    0
role            0
angkatan        0
fakultas_id     0
prodi_id        1
jurusan         1
fakultas        1
jenjang         1
dtype: int64

In [19]:
show(students_df[students_df['jenjang'] == 'S2'])

Loading ITables v2.7.0 from the internet... (need help?)


In [20]:
students_df.to_csv('data/users/students.csv', index=False)

## lecturer

In [21]:
lecturer_df = users_df[users_df['role'] == 'lecturer']
lecturer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 1785 to 1797
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ID Anggota    13 non-null     str  
 1   Nama Anggota  13 non-null     str  
 2   role          13 non-null     str  
dtypes: str(3)
memory usage: 1.0 KB


In [22]:
show(lecturer_df)

Loading ITables v2.7.0 from the internet... (need help?)


In [23]:
lecturer_data = [
    [196207282021211000, "Manajemen"],
    [196211201991032000, "Magister Ilmu Lingkungan"],
    [196509291992032000, "Akuntansi"],
    [196701231993032000, "Akuntansi"],
    [196903132021211000, ""],
    [197606272021211000, ""],
    [197708172021212000, "Desain Interior"],
    [198511062019031000, "Desain Komunikasi Visual"],
    [198706172025062000, "Ekonomi Pembangunan"],
    [199003082024061001, "Ilmu Komunikasi"],
    [199010282024062000, "Fisika"],
    [199312042024061000, "Fisika"],
    [199509302024062000, "Agribisnis"],
    [199901062024062000, "Agribisnis"],
]

In [24]:
jurusan_to_prodi = {v: k for k, v in jurusan_map.items()}

In [25]:
def extract_lecturer_data(data):
    df = pd.DataFrame(data, columns=['ID Anggota', 'jurusan'])

    df['jurusan'] = df['jurusan'].str.strip().replace('', np.nan)

    df['prodi_id'] = df['jurusan'].map(jurusan_to_prodi)
    df['fakultas_id'] = df['prodi_id'].map(prodi_to_fakultas)
    df['fakultas'] = df['fakultas_id'].map(fakultas_map)

    df['jenjang'] = df['prodi_id'].apply(
        lambda x: 'S2' if pd.notna(x) and x >= 60 and x < 70 else ('S1' if pd.notna(x) else pd.NA)
    )

    df['ID Anggota'] = (
        df['ID Anggota']
        .astype(str)
        .str.replace(',', '.', regex=False)
    )

    return df[['ID Anggota', 'prodi_id', 'jurusan', 'fakultas_id', 'fakultas','jenjang']]


In [26]:
lecturer_enrich = extract_lecturer_data(lecturer_data)
show(lecturer_enrich)

Loading ITables v2.7.0 from the internet... (need help?)


In [27]:
lecturer_df = lecturer_enrich.merge(
    lecturer_df,
    left_on='ID Anggota',
    right_on='ID Anggota',
    how='left'
)
show(lecturer_df)

Loading ITables v2.7.0 from the internet... (need help?)


In [28]:
lecturer_df.loc[lecturer_df['prodi_id'].isna(), 'role'] = 'staff'

staff_df = lecturer_df[lecturer_df['role'] == 'staff']
show(staff_df)

Loading ITables v2.7.0 from the internet... (need help?)


In [29]:
staff_df.to_csv('data/users/staff.csv', index=False)

In [30]:
lecturer_df = lecturer_df[lecturer_df['role'] == 'lecturer']
show(lecturer_df)

Loading ITables v2.7.0 from the internet... (need help?)


In [31]:
lecturer_df.to_csv('data/users/lecturer.csv', index=False)

## all users

In [32]:
users_data = pd.concat([lecturer_df, students_df, staff_df, unknown_df], ignore_index=True)
show(users_data)

Loading ITables v2.7.0 from the internet... (need help?)


In [33]:
users_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1798 entries, 0 to 1797
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID Anggota    1798 non-null   str    
 1   prodi_id      1785 non-null   object 
 2   jurusan       1785 non-null   object 
 3   fakultas_id   1786 non-null   float64
 4   fakultas      1785 non-null   str    
 5   jenjang       1785 non-null   str    
 6   Nama Anggota  1798 non-null   str    
 7   role          1798 non-null   str    
 8   angkatan      1775 non-null   float64
dtypes: float64(2), object(2), str(5)
memory usage: 250.1+ KB


In [34]:
users_data['role'].value_counts()

role
student     1775
lecturer      11
unknown       10
staff          2
Name: count, dtype: int64

In [35]:
users_data.tail()

,ID Anggota,prodi_id,jurusan,fakultas_id,fakultas,jenjang,Nama Anggota,role,angkatan
1793,378040802401,NaN,NaN,NaN,NaN,NaN,Afifta Aulia Azzahra Willistya,unknown,NaN
1794,380061003021,NaN,NaN,NaN,NaN,NaN,Dewi Khrisna Sawitri,unknown,NaN
1795,21219940116384,NaN,NaN,NaN,NaN,NaN,Amalia Hani,unknown,NaN
1796,21219960716383,NaN,NaN,NaN,NaN,NaN,Lisa Nadya Irawan,unknown,NaN
1797,23219780905464,NaN,NaN,NaN,NaN,NaN,"Dr. Fitri Sepviyanti Sumardi, Span-Ti, Subsp N...",unknown,NaN


In [36]:
users_data.to_csv('data/users/all_users.csv', index=False)